# Sinusoidal Encoding

In [ ]:
import torch

from helpers import get_device
from models.deep_learning.components import FourierPositionalEncoding as FPE
from models.deep_learning.components import SinusoidalPE
from models.deep_learning.components.embeddings.visualization import (
    plot_embedding,
    plot_embeddingdims,
)

device = get_device()

## Definition: Transformer Sinusoidal Frequencies

The Transformer defines a fixed set of frequencies that follow a geometric progression, enabling the encoding of positions across multiple scales.

$$
\omega_k = \frac{1}{\text{base}^{\frac{2k}{d_{\text{emb}}}}},\quad k = 1,2,\dots ,\frac{d_{\text{emb}}}{2}.
$$

Input
* $i \in \mathbb{Z}_+$

Parameters
* $\text{base} \in \mathbb{R}$ (typically $10000$)
* $d_{\text{emb}} \in \mathbb{N}$ (embedding dimension)

Frequencies
* $\omega = (\omega_1, \omega_2, \dots, \omega_{\frac{d_{\text{emb}}}{2}}) \in \mathbb{R}^{\frac{d_{\text{emb}}}{2}}$

Usage in Positional Encoding
$$
\text{PE}(i) = \big[ \sin(\omega_k \cdot i), \; \cos(\omega_k \cdot i) \big]_{k}, \quad i \in \mathbb{Z}_+ ,\ k = 1,2,\dots ,\frac{d_{\text{emb}}}{2} 
$$

Output
* $\text{PE}(i) \in [-1,1]^{d_{\text{emb}}}, \quad i \in \mathbb{Z}_+ $

In [ ]:
fourier_posenc = FPE(freq=FPE.transformer_frequency(24, 1)).to(device)
four_emb = fourier_posenc(spatial_dimensions=(50,))
four_emb.shape

## Embedding Visualization (per dimension)

In [ ]:
position = FPE.make_coords((50,), torch.device("cpu"))

plot_embeddingdims(
    position, four_emb.detach().cpu(), title="Fourier Positional Embedding"
)

## Sinusoidal PE

In [ ]:
sinu_pe = SinusoidalPE(d_model=24).to(device)
x = torch.zeros(50, 1).to(device)
sinu_emb = sinu_pe(x)
sinu_emb.shape

In [ ]:
torch.norm(sinu_emb - four_emb, dim=-1).max()

In [ ]:
position = FPE.make_coords((50,), torch.device("cpu"))

plot_embeddingdims(
    position, sinu_emb.detach().cpu(), title="Sinusoidal Positional Embedding"
)

## Embedding Visualization (heat map)

In the 1D case (sequence), this formulation is equivalent to the sinusoidal positional encoding used in the Transformer, as it employs the same frequency scaling.

In [ ]:
fourier_posenc = FPE(freq=FPE.transformer_frequency(130)).to(device)
emb = fourier_posenc(spatial_dimensions=(50,))

plot_embedding(emb.detach().cpu(), "Fourier(sinusoidal) Positional Embedding")

In [ ]:
sinu_pe = SinusoidalPE(d_model=130).to(device)
x = torch.zeros(50, 1).to(device)
sinu_emb = sinu_pe(x)

plot_embedding(sinu_emb.detach().cpu(), "Sinusoidal Positional Embedding")